# B2.2 · Verify signals that don't lie

**Function B — Application Security with an AI SDLC → The Harnesses that Test CyberTravels**  ·  *AI for Security*

Builds on **[B2.1 · Plan–act–verify](https://spbreed.github.io/cyber-commons/lessons/B2.1.html)**.

| | |
|---|---|
| Tools used | pytest, Checkov, GLM-4.6, Claude Sonnet 5 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

A test that passes, an exploit that fires, a compiler that accepts — these cannot be talked into agreeing with you. Model self-assessment can, and it will, because agreeing is cheaper than being right.

> **At CyberTravels.** A failing test against the booking replica cannot be talked into agreeing. The model's own assessment of its patch can, and will.

## 2 · The framework

```
   signals that cannot be talked into agreeing

   test suite exit code      compiler output      exploit fires
   +----------------+        +-------------+      +-----------+
   |  0 / non-zero  |        | ok / error  |      | yes / no  |
   +----------------+        +-------------+      +-----------+

   signals that can
   +------------------------------------------------+
   | "I have verified the fix is correct."  (rating) |
   +------------------------------------------------+
```

This is the highest-value lesson in the track, because every other control
assumes the verifier is honest.

Verifiers form a hierarchy, ordered by **what it takes to fool them**:

| Verifier | Fooled by | Available when |
|---|---|---|
| **Behavioural test** | changing real behaviour | you can execute the thing |
| **Exact-match oracle** | nothing, but needs the answer up front | rarely |
| **Shape check** | any well-formed output | always |
| **LLM judge** | confident prose | always |

The trap is that the two available-everywhere options are the two weakest, and
they fail in the worst possible direction: they do not error, they **approve**.

There is also a subtler failure that is worth seeing rather than reading about:
a verifier that is *correct* but reads stale state. A test runner that imports
cached bytecode reports on code that is no longer on disk. A lying oracle is
worse than no oracle, because you stop looking.

## 3 · Demo — four verifiers, one broken input

In [ ]:
BROKEN  = "def parse_port(s): return int(s)"          # accepts 0, 99999, -1
CORRECT = ("def parse_port(s):\n"
           "    p = int(s)\n"
           "    if not (1 <= p <= 65535): raise ValueError('port out of range')\n"
           "    return p")

def behavioural(src):
    ns = {}
    try:
        exec(compile(src, "<p>", "exec"), ns); fn = ns["parse_port"]
    except Exception as e:
        return False, f"compile failed: {e}"
    for bad in ("0", "70000", "-1"):
        try:
            fn(bad)
            return False, f"accepted out-of-range port {bad!r}"
        except ValueError:
            pass
    try:
        if fn("443") != 443:
            return False, "rejected a valid port"
    except Exception as e:
        return False, f"valid port raised {e}"
    return True, "rejects out-of-range, accepts valid"

def exact_match(expected):
    return lambda src: (src.strip() == expected.strip(),
                        "exact match" if src.strip() == expected.strip()
                        else "differs from the reference implementation")

def shape_check(src):
    ok = src.strip().startswith("def parse_port")
    return ok, "defines parse_port" if ok else "wrong shape"

def llm_judge(src):
    ok = bool(src.strip()) and not src.lower().startswith("i cannot")
    return ok, "judge: this looks like a correct implementation"

VERIFIERS = {"behavioural (executes it)": behavioural,
             "exact-match oracle":        exact_match(CORRECT),
             "shape check":               shape_check,
             "llm judge":                 llm_judge}

print(f"{'verifier':28s}{'on BROKEN':12s}{'on CORRECT':12s}detail (broken)")
print("-" * 84)
for name, v in VERIFIERS.items():
    b_ok, b_why = v(BROKEN)
    c_ok, _     = v(CORRECT)
    print(f"{name:28s}{str(b_ok):12s}{str(c_ok):12s}{b_why[:32]}")

## 4 · Where it breaks — the exact-match oracle is also wrong

Look at the `on CORRECT` column. The behavioural verifier is the only one that gets *both* right. The exact-match oracle rejects a correct implementation that differs from its reference — which is why nobody uses it, and why teams fall back to the two weak options.

In [ ]:
ALTERNATIVE = ("def parse_port(s):\n"
               "    p = int(s)\n"
               "    if p < 1 or p > 65535:\n"
               "        raise ValueError('bad port')\n"
               "    return p")
print("a correct implementation, written differently:")
for name, v in VERIFIERS.items():
    ok, why = v(ALTERNATIVE)
    print(f"   {name:28s}{str(ok):7s}{why[:44]}")
print("\nThe oracle says no. Behavioural says yes. Only one of those is useful")
print("on code you did not write in advance.")

## 5 · The subtler failure — a correct verifier reading stale state

This one is not about weak checks. The check is right; the *input* to it is stale. Python's bytecode cache reproduces it faithfully.

In [ ]:
import os, sys, tempfile, subprocess, textwrap, shutil, pathlib

work = pathlib.Path(tempfile.mkdtemp())
(work / "mod.py").write_text("def check(x):\n    return True   # broken: always passes\n")
(work / "test_mod.py").write_text(textwrap.dedent("""
    from mod import check
    def test_rejects_bad():
        assert check(-1) is False
"""))

def run_check(workdir, clear_cache):
    if clear_cache:
        shutil.rmtree(workdir / "__pycache__", ignore_errors=True)
    env = {**os.environ, "PYTHONPATH": str(workdir)}
    if clear_cache:
        env["PYTHONDONTWRITEBYTECODE"] = "1"
    r = subprocess.run([sys.executable, "-c",
                        "import mod; print('PASS' if mod.check(-1) is False else 'FAIL')"],
                       cwd=workdir, env=env, capture_output=True, text=True)
    return r.stdout.strip()

print("1. verifier on the broken code:      ", run_check(work, clear_cache=True))
# the agent "fixes" it
(work / "mod.py").write_text("def check(x):\n    return x >= 0\n")
print("2. after a real fix, cache cleared:  ", run_check(work, clear_cache=True))
# now put the broken version back, but leave a stale cache in place
import py_compile
(work / "mod.py").write_text("def check(x):\n    return True   # broken again\n")
py_compile.compile(str(work / "mod.py"), doraise=True)
(work / "mod.py").write_text("def check(x):\n    return x >= 0\n")
os.utime(work / "mod.py", (0, 0))          # make the source look older than the cache
print("3. source fixed, STALE cache honoured:", run_check(work, clear_cache=False),
      " ← the verifier is reading code that is not on disk")
shutil.rmtree(work, ignore_errors=True)

## 6 · The control — rank verifiers and always clear derived state

In [ ]:
RANKING = [
 (1, "behavioural / property test", "must change observable behaviour",
     "execute the artefact against facts that must hold"),
 (2, "differential test",           "must match a trusted second implementation",
     "run old and new against the same inputs"),
 (3, "exact-match oracle",          "nothing — but needs the answer in advance",
     "only usable on a fixed corpus"),
 (4, "shape / schema check",        "any well-formed output",
     "use for conformance ONLY, never for quality — see B2.11"),
 (5, "llm judge",                   "confident prose",
     "acceptable only as a filter before a real check, never as the last word"),
]
print(f"{'rank':5s}{'verifier':30s}{'fooled by':44s}")
print("-" * 80)
for r, name, fooled, use in RANKING:
    print(f"{r:<5}{name:30s}{fooled:44s}")
    print(f"{'':35s}{use}")

CHECKLIST = [
 "does it EXECUTE the artefact, or only inspect it?",
 "would it fail if the artefact were subtly wrong?",
 "does it clear caches / derived state before reading?",
 "is its own correctness tested (does it fail on known-bad input)?",
]
print("\nverifier review checklist:")
for c in CHECKLIST:
    print("   ·", c)

# the fourth item, applied to our own verifier
assert behavioural(BROKEN)[0] is False, "verifier must fail on known-bad"
assert behavioural(CORRECT)[0] is True, "verifier must pass on known-good"
assert behavioural(ALTERNATIVE)[0] is True, "verifier must accept alternatives"
print("\nour behavioural verifier passes its own test: fails broken, accepts both correct forms.")

## What you just proved

Only the behavioural verifier gets both the broken and correct inputs right; the shape check and judge accept the broken port parser, and the exact-match oracle rejects a correct alternative implementation. The stale-cache demo shows the verifier reporting PASS while reading bytecode for code no longer on disk. The verifier's own test confirms it fails on known-bad and accepts both correct forms.

## Your turn

Find one check in your pipeline that is a shape check wearing an oracle's name — "the build passed", "the JSON validated", "no errors in the log". Then ask the fourth checklist question about it: has anyone ever confirmed it fails on known-bad input?

---

**Next → [B2.3 · Tool design](https://spbreed.github.io/cyber-commons/lessons/B2.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*